# Fine-tune YOLO11s for Chilean licence plates
Run this notebook with a Colab GPU. It keeps the original dataset read-only, builds group-aware splits, trains YOLO11s, evaluates the held-out test split, and exports ONNX.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update these two paths for your Drive layout.
REPO_ROOT = '/content/drive/MyDrive/Chillean_object_detection'
DATASET_ROOT = f'{REPO_ROOT}/dataset2-20260922T180035Z-1-001/dataset2'
PREPARED_ROOT = '/content/prepared_dataset'
RUNS_ROOT = f'{REPO_ROOT}/runs/license_plate'

In [ ]:
!pip install -q -r "{REPO_ROOT}/training/requirements-colab.txt"

In [ ]:
!python "{REPO_ROOT}/training/prepare_dataset.py" --dataset-root "{DATASET_ROOT}" --manifest "{REPO_ROOT}/training/manifest.csv" --transcriptions "{REPO_ROOT}/training/transcriptions.csv" --materialize "{PREPARED_ROOT}"

Before training, open `training/transcriptions.csv`, visually verify the inferred plate strings, and fill the blank strings. These strings are used only for OCR evaluation; YOLO training uses the bounding boxes.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Switch Colab to a GPU runtime before training.'
print(torch.cuda.get_device_name(0))
!python "{REPO_ROOT}/training/train.py" --data "{PREPARED_ROOT}/data.yaml" --project "{RUNS_ROOT}" --device 0

In [ ]:
BEST = f'{RUNS_ROOT}/yolo11s_plates/weights/best.pt'
EXPORT_DIR = f'{REPO_ROOT}/artifacts/models'
!python "{REPO_ROOT}/training/export_onnx.py" --weights "{BEST}" --output-dir "{EXPORT_DIR}"

Copy `artifacts/models/plate_detector_fp32.onnx` and `model_metadata.json` into `mobile/android/app/src/main/assets/models/`, then build the Android app.